# Black-Scholes-Merton Methodology

This notebook documents the calculations used in Black-Scholes Options Lab. It is intentionally concise: the Streamlit app is the main interface, and this notebook keeps the mathematical assumptions and example outputs easy to audit.

## Model Assumptions

Black-Scholes-Merton prices European-style options under a simplified market model:

- The underlying price follows a lognormal process.
- Volatility is constant over the option's life.
- The risk-free rate is constant.
- Dividend yield, when used, is modeled as a continuous yield.
- Markets are frictionless, with no transaction costs or liquidity constraints.
- Exercise occurs only at expiration.

## Pricing Formula

For spot price `S`, strike `K`, time to expiration `T`, rate `r`, dividend yield `q`, and volatility `sigma`:

```text
Call = S e^(-qT) N(d1) - K e^(-rT) N(d2)
Put  = K e^(-rT) N(-d2) - S e^(-qT) N(-d1)

d1 = [ln(S/K) + (r - q + sigma^2 / 2)T] / [sigma sqrt(T)]
d2 = d1 - sigma sqrt(T)
```

The app also handles expiration and zero-volatility edge cases by falling back to intrinsic or discounted forward intrinsic value.

In [ ]:
from black_scholes_options_lab.greeks import calculate_greeks
from black_scholes_options_lab.pricing import option_summary

spot = 100
strike = 100
time_to_expiry = 1
risk_free_rate = 0.05
volatility = 0.20

summary = option_summary(spot, strike, time_to_expiry, risk_free_rate, volatility, "call")
greeks = calculate_greeks(spot, strike, time_to_expiry, risk_free_rate, volatility, "call")

summary, greeks

## Greeks

- Delta measures sensitivity to a $1 move in the underlying stock.
- Gamma measures the curvature of delta.
- Theta measures time decay. The dashboard displays theta per day.
- Vega measures sensitivity to volatility. The dashboard displays vega per 1 volatility percentage point.
- Rho measures sensitivity to rates. The dashboard displays rho per 1 interest-rate percentage point.

In [ ]:
call_greeks = calculate_greeks(100, 100, 1, 0.05, 0.20, "call")
put_greeks = calculate_greeks(100, 100, 1, 0.05, 0.20, "put")

{
    "call_delta": call_greeks.delta,
    "put_delta": put_greeks.delta,
    "shared_gamma": call_greeks.gamma,
    "theta_per_day": call_greeks.theta_per_day,
    "vega_per_1pct": call_greeks.vega_per_percent,
    "rho_per_1pct": call_greeks.rho_per_percent,
}

## Historical Volatility

Historical volatility is calculated from daily log returns:

```text
r_t = ln(P_t / P_{t-1})
annualized volatility = standard_deviation(r_t) * sqrt(252)
```

This is a backward-looking estimate. It is not implied volatility from the option chain and should be treated as an assumption, not a forecast.

In [ ]:
import pandas as pd

from black_scholes_options_lab.volatility import (
    historical_volatility,
    rolling_volatility,
)

prices = pd.Series([100, 102, 101, 105, 107, 104], dtype=float)
historical_volatility(prices), rolling_volatility(prices, window=3).dropna().tail()

## Sensitivity Analysis

The dashboard recalculates option value while changing one or two assumptions at a time. One-dimensional charts isolate a single input, while heatmaps show how value changes across stock price and volatility or stock price and time to expiration.

In [ ]:
import numpy as np

from black_scholes_options_lab.sensitivity import option_value_curve

option_value_curve(
    "spot",
    np.array([90, 100, 110]),
    spot=100,
    strike=100,
    time_to_expiry=1,
    risk_free_rate=0.05,
    volatility=0.20,
    option_type="call",
)

## Limitations

Black-Scholes-Merton is transparent, but simplified. Real option prices can reflect early exercise features, discrete dividends, volatility smiles, changing rates, bid-ask spreads, liquidity, transaction costs, and market supply-demand effects that are outside the standard model. Outputs from this project are educational estimates, not investment advice or trade recommendations.